# 

Olivier V. Raven [](https://orcid.org/0009-0008-6184-6831) (University of Waikato)  
Calum MacNeil [](https://orcid.org/0000-0003-0402-9292) (The Cawthron Institute)  
Robin Holmes [](https://orcid.org/0000-0002-8737-2717) (The Cawthron Institute)  
Ian A. Kusabs [](https://orcid.org/0000-0002-1473-7447) (Ian Kusabs and Associates Limited)  
Francis J. Burdon [](https://orcid.org/0000-0002-5398-4993) (University of Waikato)  
Deniz Özkundakci [](https://orcid.org/0000-0002-5442-4576) (University of Waikato)  
Invalid Date

# Introduction

# Materials and Methods

# Results

## Setup

In [ ]:
packages <- c("XNomial","brms","nnet","lme4","multcompView","patchwork","janitor","lubridate","stringr","tidyverse","dplyr","ggplot2","readxl","writexl","readr")

quiet_load <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    suppressWarnings(suppressMessages(install.packages(pkg, dependencies = TRUE)))}
  suppressPackageStartupMessages(require(pkg, character.only = TRUE, quietly = TRUE))
  invisible(TRUE)}

invisible(lapply(packages, quiet_load))

# Set paths
exc_file_dir    <- "data/raw/Mesocosm_experiment.xlsx"
raw_data_dir    <- "data/raw"
der_data_dir    <- "data/derived"
out_dir         <- "outputs"

read_sheet <- function(sheet) {readxl::read_excel(exc_file_dir, sheet = sheet)}
write_out <- function(df, name) {write.csv(df, file.path(der_data_dir, name), row.names = FALSE)}

# Read data
df_koura               <- read_sheet("Koura")
df_setup               <- read_sheet("Tank_setup")
df_exp1_single_raw     <- read_sheet("Reef_single")
df_exp2_group_raw      <- read_sheet("Reef_group")
df_exp3_woodstone_raw  <- read_sheet("Stone_wood")
df_exp4_brick_raw      <- read_sheet("Bricks")
df_wq                  <- read_sheet("WQ")
df_loggers             <- read_sheet("Loggers")

# Save as csv to derived
write_out(df_koura,              "df_koura.csv")
write_out(df_setup,              "df_setup.csv")
write_out(df_exp1_single_raw,    "df_exp1_single_raw.csv")
write_out(df_exp2_group_raw,     "df_exp2_group_raw.csv")
write_out(df_exp3_woodstone_raw, "df_exp3_woodstone_raw.csv")
write_out(df_exp4_brick_raw,     "df_exp4_brick_raw.csv")
write_out(df_wq,                 "df_wq.csv")
write_out(df_loggers,            "df_loggers.csv")

# set preferred orders
REEF_LEVELS  <- c("SingleStone","FlatStone","StonePile","WoodLog","TankWall")
REEF_LEVELS3 <- c("FlatStone","WoodSplit","TankWall", "NA")
REEF_LEVELS4 <- c("BrickArtificial","BrickWood","TankWall")
LOC_LEVELS   <- c("A", "B", "C", "D", "E")
LOC_LEVELS3  <- c("A", "B", "E")

relevel_reef  <- function(df) df %>% mutate(reef_type = factor(as.character(reef_type),levels = REEF_LEVELS))
relevel_reef3 <- function(df) df %>% mutate(reef_type = factor(as.character(reef_type),levels = REEF_LEVELS3))
relevel_reef4 <- function(df) df %>% mutate(reef_type = factor(as.character(reef_type),levels = REEF_LEVELS4))

# Set plot theme
base_theme_bw <- theme_classic() +
  theme(text     = element_text(family = "Arial", size = 11),
    axis.title   = element_text(face = "plain"),
    axis.text    = element_text(face = "plain"),
    plot.title   = element_text(face = "plain"),
    strip.text   = element_text(face = "plain"),
    panel.border = element_rect(colour = "black", fill = NA, linewidth = 0.5))
theme_set(base_theme_bw)

## Make DF’s into long format

In [ ]:
df_exp1_single_long <- df_exp1_single_raw %>%
  mutate(location_l0 = l0, location_l1 = l1) %>%
  pivot_longer(cols = matches("_(l0|l1)$"),
    names_to = c(".value", "stage"),
    names_pattern = "^(.*)_(l[01])$") %>%
  mutate(stage         = toupper(stage),
    location_code = factor(toupper(location), levels = LOC_LEVELS),
    reef_type     = factor(reef, levels = REEF_LEVELS)) %>%
  select(tank, round, group_id, koura_id, size_class, sex, size_mm1, weight_g1, date, stage, t_leave_s, t_l0_s, location_code, reef_type,legs1, claws1, antenna1, size_mm2, weight_g2, legs2, claws2, antenna2,notes_exp, notes_koura)

write_out(df_exp1_single_long, "df_exp1_single_long.csv")

# EXP 2 – Group reefs (long)
df_exp2_group_long <- df_exp2_group_raw %>%
  mutate(location_l0 = l0, location_l1 = l1) %>%
  pivot_longer(cols = c(location_l0, location_l1,
      reef_l0, reef_l1,legs_l0, claws_l0, antenna_l0,legs_l1, claws_l1, antenna_l1), names_to = c(".value", "stage"), names_pattern = "^(.*)_(l[01])$") %>%
  mutate(stage          = toupper(stage),
    location_code  = factor(toupper(location), levels = LOC_LEVELS),
    reef_type      = factor(reef, levels = REEF_LEVELS),
    size_mm_stage  = if_else(stage == "L0", size_mm1, size_mm2, NA_real_),
    weight_g_stage = if_else(stage == "L0", weight_g1, weight_g2, NA_real_)) %>%
  select(tank, round, group_id, koura_id, size_class, sex, date,stage, t_leave_s, t_l0_s, location_code, reef_type,legs, claws, antenna, size_mm_stage, weight_g_stage,size_mm1, weight_g1, legs1, claws1, antenna1,size_mm2, weight_g2, legs2, claws2, antenna2,notes_exp, notes_koura)

write_out(df_exp2_group_long, "df_exp2_group_long.csv")


# EXP 3 – Stone / wood (long)
df_exp3_woodstone_long <- df_exp3_woodstone_raw %>%
  mutate(location_l0 = l0, location_l1 = l1) %>%
  pivot_longer(cols = c(location_l0, location_l1, reef_l0, reef_l1),
    names_to = c(".value", "stage"),
    names_pattern = "^(.*)_(l[01])$") %>%
  mutate(stage         = toupper(stage),
    location_code = factor(toupper(location), levels = LOC_LEVELS3),
    reef_type     = factor(reef, levels = REEF_LEVELS3)) %>%
  select(tank, round, group_id, koura_id, size_class, sex, date, stage, location_code, reef_type, size_mm1, weight_g1, legs1, claws1, antenna1, size_mm2, weight_g2, legs2, claws2, antenna2, notes_exp, notes_koura)

write_out(df_exp3_woodstone_long, "df_exp3_woodstone_long.csv")


# EXP 4 – Bricks (long)
df_exp4_brick_long <- df_exp4_brick_raw %>%
  mutate(location_l1 = l1) %>%
  pivot_longer(cols = c(location_l1, reef_l1),
    names_to = c(".value", "stage"),
    names_pattern = "^(.*)_(l[01])$") %>%
  mutate(stage         = toupper(stage),
    location_code = factor(toupper(location), levels = LOC_LEVELS),
    reef_type     = factor(reef, levels = REEF_LEVELS4)) %>%
  select(tank, round, group_id, koura_id, size_class, sex, date, stage, location_code, reef_type,size_mm1, weight_g1, legs1, claws1, antenna1,size_mm2, weight_g2, legs2, claws2, antenna2,notes_exp, notes_koura)

write_out(df_exp4_brick_long, "df_exp4_brick_long.csv")

## Exp 1 Single koura

### Exp 1 I

In [ ]:
exp1_reef_combined <- df_exp1_single_long %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, name = "n") %>%
  group_by(stage) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef() %>%
  mutate(panel = "Combined")

exp1_reef_bysex <- df_exp1_single_long %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, sex, name = "n") %>%
  group_by(stage, sex) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef() %>%
  mutate(panel = dplyr::recode(sex, F = "Female", M = "Male")) %>%
  select(-sex)

exp1_reef_all <- dplyr::bind_rows(exp1_reef_combined, exp1_reef_bysex)

lab_stage <- c(L0 = "Initial location (L0)", L1 = "Next-day location (L1)")
lab_panel1 <- c(Combined = "Combined", Female = "Female", Male = "Male")

# One plot ADD: n values above each bar and in 
Exp1_reef_p <- exp1_reef_all %>%
  relevel_reef() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage, labeller = labeller(stage = lab_stage, panel = lab_panel1)) +
  labs(x = "Reef type", y = "Proportion")

ggsave(Exp1_reef_p, file = file.path(out_dir, "Exp1_reef_p.png"), width = 8, height = 6, dpi = 300)

systemfonts and textshaping have been compiled with different versions of Freetype. Because of this, textshaping will not use the font cache provided by systemfonts

Warning in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)): font family
not found in Windows font database
Warning in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)): font family
not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

### Exp 1 II

In [ ]:
invisible(capture.output({
  # proportions by x var and facet var (per stage)
  mk_props <- function(data, xvar, facetvar) {
    data %>%
      dplyr::filter(stage %in% c("L0","L1")) %>%
      dplyr::count(stage, {{ xvar }}, {{ facetvar }}, name = "n") %>%
      dplyr::group_by(stage, {{ facetvar }}) %>%
      dplyr::mutate(N = sum(n), prop = n / N) %>%
      dplyr::ungroup()
  }

  # bar plot; rows = facetvar, cols = stage; fixed y-scale across all plots
  mk_bar <- function(df, xvar, facetvar, title, reef = FALSE) {
    p <- ggplot(df, aes({{ xvar }}, prop)) +
      geom_col(color = "black") +
      geom_text(aes(label = n), vjust = 1.3, colour = "white") +
      facet_grid(rows = vars({{ facetvar }}), cols = vars(stage)) +
      labs(title = title) +
      scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.05)))
    if (reef) {
      p <- p +
        scale_x_discrete(limits = REEF_LEVELS, drop = FALSE) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))
    }
    p
  }

  # data
  exp1_loc    <- mk_props(df_exp1_single_long, location_code, tank)
  exp1_round  <- mk_props(df_exp1_single_long, location_code, round)
  exp1_loc2   <- mk_props(df_exp1_single_long, reef_type,     tank)
  exp1_round2 <- mk_props(df_exp1_single_long, reef_type,     round)

  # plots
  p_exp1_loc    <- mk_bar(exp1_loc,   location_code, tank,  "Location by tank")
  p_exp1_round  <- mk_bar(exp1_round, location_code, round, "Location by round")
  p_exp1_loc2   <- mk_bar(exp1_loc2,  reef_type,     tank,  "Reeftype by tank",  reef = TRUE)
  p_exp1_round2 <- mk_bar(exp1_round2,reef_type,     round, "Reeftype by round", reef = TRUE)

  loc_round_plots_exp1 <<- p_exp1_loc + p_exp1_loc2 + p_exp1_round + p_exp1_round2 +
    patchwork::plot_layout(ncol = 4)
}))
loc_round_plots_exp1

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call(C_stringMetric, as.graphicsAnnot(x$label)): font family
not found in Windows font database

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBound

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

...

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

### Exp 1 III

In [ ]:
exp1_pairs <- df_exp1_single_long %>%
  dplyr::filter(stage %in% c("L0","L1")) %>%
  dplyr::select(koura_id, tank, round, size_class, sex, stage,location_code, reef_type, t_leave_s, t_l0_s) %>%
  dplyr::mutate(
    location_code = forcats::fct_drop(factor(location_code, levels = LOC_LEVELS)),
    reef_type     = forcats::fct_drop(factor(reef_type,     levels = REEF_LEVELS)),
    t_leave_s     = suppressWarnings(as.numeric(t_leave_s)),
    t_l0_s        = suppressWarnings(as.numeric(t_l0_s))) %>%
  tidyr::pivot_wider(names_from = stage,values_from = c(location_code, reef_type, t_leave_s, t_l0_s),names_sep = "_")

activity_df <- exp1_pairs %>%
  dplyr::transmute(koura_id, tank, round, size_class, sex, time_to_leave = t_leave_s_L0, time_at_L0 = t_l0_s_L0) 

activity_long <- activity_df %>%
  mutate(
    time_to_leave = suppressWarnings(as.numeric(time_to_leave)),time_at_L0    = suppressWarnings(as.numeric(time_at_L0))) %>%
  pivot_longer(c(time_to_leave, time_at_L0), names_to = "metric", values_to = "time") %>%
  filter(!is.na(time), time >= 0)

lab_metric <- c(time_to_leave = "Time to leave start (t_leave)",
                time_at_L0    = "Time to reach initial hide (t_L0)")

act_exp1_p <- ggplot(activity_long, aes(time, col=sex)) +
  stat_ecdf(geom = "step") +
  facet_grid( ~ metric, labeller = labeller(metric = lab_metric)) +
  labs(x = "Time (s)", y = "Proportion completed by time")

act_exp1_p

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database

### Exp 1 Statistic

In [ ]:
# RQ1: 
# H0:
# H1: 

# RQ2: Is there a statistical difference between reef type between L0 vs L1? 
# H0:
# H1: 


df1 <- df_exp1_single_long %>%
  mutate(
    reef_type      = factor(reef_type),
    stage          = factor(stage, levels = c("L0", "L1")),
    sex            = factor(sex),
    size_class     = factor(size_class),
    location_code  = factor(location_code),
    tank           = factor(tank),
    round          = factor(round),
    koura_id       = factor(koura_id),
    size_mm1       = as.numeric(size_mm1),
    weight_g1      = as.numeric(weight_g1)) %>%
  filter(!is.na(reef_type))

# most preferred reef at L1
df1_L1 <- df1 %>% filter(stage == "L1")

tab_L1 <- table(df1_L1$reef_type)
tab_L1_nz <- tab_L1[tab_L1 > 0]        
XNomial::xmulti(obs  = as.numeric(tab_L1_nz), expr = rep(sum(tab_L1_nz) / length(tab_L1_nz), length(tab_L1_nz)))


P value (LLR) = 1.57e-05

# weights:  25 (16 variable)
initial  value 96.566275 
iter  10 value 69.102839
iter  20 value 67.050810
iter  30 value 66.281166
iter  40 value 66.158182
iter  50 value 66.094016
iter  60 value 66.047753
final  value 66.031211 
converged

# weights:  20 (12 variable)
initial  value 96.566275 
iter  10 value 75.289762
iter  20 value 73.621116
iter  30 value 73.193816
iter  40 value 72.992209
iter  50 value 72.876080
iter  60 value 72.850567
iter  70 value 72.844162
final  value 72.840227 
converged

Call:
nnet::multinom(formula = reef_type ~ stage + sex + size_mm1, 
    data = df1)

Coefficients:
          (Intercept)    stageL1       sexM  size_mm1
FlatStone   -20.05042  0.2548323  0.4994737 0.8074787
StonePile   -13.51237  0.2975835 -0.2076332 0.5685100
WoodLog     -10.16247  2.0974287  0.6306893 0.4217070
TankWall    -16.29879 -8.8580967  8.4281415 0.3289003

Std. Errors:
          (Intercept)     stageL1      sexM  size_mm1
FlatStone    9.640022 1.352491920  1.245280 0.3737658
StonePile    9.450350 1.349945525  1.249425 0.3689399
WoodLog      9.076447 1.256769165  1.180059 0.3557096
TankWall    44.266128 0.001846838 42.216347 0.5185328

Residual Deviance: 132.0624 
AIC: 164.0624 

Call:
nnet::multinom(formula = reef_type ~ sex + size_mm1, data = df1)

Coefficients:
          (Intercept)       sexM  size_mm1
FlatStone  -19.671087  0.4904121 0.7955914
StonePile  -13.257669 -0.2119649 0.5616943
WoodLog     -9.459348  0.6005729 0.4352346
TankWall   -14.305730  7.1521973 0.2891267

Std. Errors:
          (Intercept)      sexM  size_mm1
FlatStone    9.431989  1.228230 0.3656477
StonePile    9.282664  1.239767 0.3621961
WoodLog      8.686581  1.132196 0.3415133
TankWall    25.854308 22.358166 0.5062360

Residual Deviance: 145.6805 
AIC: 169.6805 

, , FlatStone

                   2.5 %    97.5 %
(Intercept) -38.94451648 -1.156325
stageL1      -2.39600314  2.905668
sexM         -1.94123010  2.940178
size_mm1      0.07491117  1.540046

, , StonePile

                  2.5 %   97.5 %
(Intercept) -32.0347121 5.009981
stageL1      -2.3482611 2.943428
sexM         -2.6564609 2.241195
size_mm1     -0.1545989 1.291619

, , WoodLog

                  2.5 %   97.5 %
(Intercept) -27.9519767 7.627044
stageL1      -0.3657936 4.560651
sexM         -1.6821830 2.943562
size_mm1     -0.2754710 1.118885

, , TankWall

                   2.5 %    97.5 %
(Intercept) -103.0588078 70.461224
stageL1       -8.8617165 -8.854477
sexM         -74.3143787 91.170662
size_mm1      -0.6874053  1.345206

Likelihood ratio tests of Multinomial Models

Response: reef_type
                   Model Resid. df Resid. Dev   Test    Df LR stat.     Pr(Chi)
1         sex + size_mm1       228   145.6805                                  
2 stage + sex + size_mm1       224   132.0624 1 vs 2     4 13.61803 0.008619424

## Exp 2 Group koura

### Exp 2 I

In [ ]:
exp2_reef_combined <- df_exp2_group_long %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, name = "n") %>%
  group_by(stage) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef() %>%
  mutate(panel = "Combined")

# By size class
exp2_reef_bysize <- df_exp2_group_long %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, size_class, name = "n") %>%
  group_by(stage, size_class) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef() %>%
  mutate(panel = dplyr::recode(as.character(size_class), S = "Small", M = "Medium", L = "Large")) %>%
  select(-size_class)

# Bind for one facetted plot
exp2_reef_all <- dplyr::bind_rows(exp2_reef_combined, exp2_reef_bysize)

# Labels 
lab_panel2 <- c(Combined = "Combined", Small = "Small", Medium = "Medium", Large = "Large")

# One plot
Exp2_reef_p <- exp2_reef_all %>%
  relevel_reef() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage, labeller = labeller(stage = lab_stage, panel = lab_panel2)) +
  labs(x = "Reef type", y = "Proportion")

ggsave(Exp2_reef_p, file = file.path(out_dir, "Exp2_reef_p.png"), width = 8, height = 8, dpi = 300)

Exp2_reef_p

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

### Exp 2 II

In [ ]:
invisible(capture.output({

  # proportions by x var and facet var (per stage)
  mk_props <- function(data, xvar, facetvar) {
    data %>%
      dplyr::filter(stage %in% c("L0","L1")) %>%
      dplyr::count(stage, {{ xvar }}, {{ facetvar }}, name = "n") %>%
      dplyr::group_by(stage, {{ facetvar }}) %>%
      dplyr::mutate(N = sum(n), prop = n / N) %>%
      dplyr::ungroup()
  }

  # bar plot; uses facet_grid for rows=facetvar, cols=stage
  mk_bar <- function(df, xvar, facetvar, title, reef = FALSE) {
    p <- ggplot(df, aes({{ xvar }}, prop)) +
      geom_col(color = "black") +
      geom_text(aes(label = n), vjust = 1.3, colour = "white") +
      facet_grid(rows = vars({{ facetvar }}), cols = vars(stage)) +
      labs(title = title)+
      scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.05)))
    if (reef) {
      p <- p +
        scale_x_discrete(limits = REEF_LEVELS, drop = FALSE) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))
    }
    p
  }

  # data
  exp2_loc    <- mk_props(df_exp2_group_long, location_code, tank)
  exp2_round  <- mk_props(df_exp2_group_long, location_code, round)
  exp2_loc2   <- mk_props(df_exp2_group_long, reef_type,     tank)
  exp2_round2 <- mk_props(df_exp2_group_long, reef_type,     round)

  # plots
  p_exp2_loc    <- mk_bar(exp2_loc,   location_code, tank,  "Location by tank")
  p_exp2_round  <- mk_bar(exp2_round, location_code, round, "Location by round")
  p_exp2_loc2   <- mk_bar(exp2_loc2,  reef_type,     tank,  "Reeftype by tank",  reef = TRUE)
  p_exp2_round2 <- mk_bar(exp2_round2,reef_type,     round, "Reeftype by round", reef = TRUE)

  loc_round_plots_exp2 <<- p_exp2_loc + p_exp2_loc2 + p_exp2_round + p_exp2_round2 +
    patchwork::plot_layout(ncol = 4)
}))
loc_round_plots_exp2

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBound

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

...

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

### Exp 2 III

In [ ]:
exp2_reef_koura <- df_exp2_group_long %>%
  mutate(stage = toupper(stage)) %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, koura_id, name = "n") %>%
  group_by(stage, koura_id) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef()

p_exp2_reef_koura <- exp2_reef_koura %>%
  relevel_reef() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  facet_wrap(koura_id ~ stage, nrow = 6) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

p_exp2_reef_koura

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

### Exp 2 Statistic

In [ ]:
# RQ1: Is there a statistical difference between reef type in L1 between the size classes while controlling for group_id, tank and round?
# H0:
# H1: 

# RQ2: Is there a statistical difference between reef type between L0 vs L1? 
# H0:
# H1: 

df2 <- df_exp2_group_long %>%
  mutate(
    reef_type      = factor(reef_type),
    stage          = factor(stage, levels = c("L0", "L1")),
    sex            = factor(sex), # exclude as all male
    size_class     = factor(size_class, levels = c("S", "M", "L")),
    location_code  = factor(location_code),
    tank           = factor(tank),
    round          = factor(round),
    group_id       = factor(group_id), # each group of 6 koura used 5 times
    koura_id       = factor(koura_id)) %>%
  filter(!is.na(reef_type))

form_exp2 <- reef_type ~ stage + size_class + (1 | group_id) + (1 | tank) + (1 | round)

#m_exp2 <- brm(formula = form_exp2, data = df2, family = categorical(link = "logit"), chains= 4, cores = 4,  iter = 4000, seed = 1)

#summary(m_exp2)

## Exp 3 Wood vs stone

### Exp 3 I

In [ ]:
exp3_reef <- df_exp3_woodstone_long %>%
  filter(stage %in% c("L0","L1"), !is.na(reef_type)) %>%
  count(stage, reef_type, name = "n") %>%
  group_by(stage) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>% relevel_reef3()

exp3_reef_size <- df_exp3_woodstone_long %>%
  filter(stage %in% c("L0","L1"), !is.na(reef_type)) %>%
  count(stage, reef_type, size_class, name = "n") %>%
  group_by(stage, size_class) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>% relevel_reef3()

# Panel labels and order
PANEL_LEVELS <- c("Combined","L","M","S")
lab_panel3    <- c(Combined = "Combined", L = "Large", M = "Medium", S = "Small")

# Build combined counts
exp3_top <- exp3_reef %>%
  mutate(panel = factor("Combined", levels = PANEL_LEVELS))

exp3_bot <- exp3_reef_size %>%
  mutate(panel = factor(as.character(size_class), levels = PANEL_LEVELS))

exp3_all <- bind_rows(exp3_top, exp3_bot)

# One plot with a left strip for every row (Combined, S, M, L)
Exp3_reef_p <- exp3_all %>%
  #dplyr::left_join(exp3_all_L, by = c("stage","reef_type","panel")) %>% relevel_reef3() %>% 
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +  
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage, labeller = labeller(stage = lab_stage, panel = lab_panel3)) +
  labs(x = "Reef type", y = "Proportion")

ggsave(Exp3_reef_p, file = file.path(out_dir, "Exp3_reef_p.png"), width = 8, height = 8, dpi = 300)

Exp3_reef_p

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

### Exp 3 II

In [ ]:
invisible(capture.output({
  # proportions by x var and facet var (per stage)
  mk_props <- function(data, xvar, facetvar) {
    data %>%
      dplyr::filter(stage %in% c("L0","L1")) %>%
      dplyr::count(stage, {{ xvar }}, {{ facetvar }}, name = "n") %>%
      dplyr::group_by(stage, {{ facetvar }}) %>%
      dplyr::mutate(N = sum(n), prop = n / N) %>%
      dplyr::ungroup()
  }

# bar plot; uses facet_grid for rows=facetvar, cols=stage
  mk_bar <- function(df, xvar, facetvar, title, reef = FALSE) {
    p <- ggplot(df, aes({{ xvar }}, prop)) +
      geom_col(color = "black") +
      geom_text(aes(label = n), vjust = 1.3, colour = "white") +
      facet_grid(rows = vars({{ facetvar }}), cols = vars(stage)) +
      labs(title = title)+
      scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.05)))
    if (reef) {
      p <- p +
        scale_x_discrete(limits = REEF_LEVELS3, drop = FALSE) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))
    }
    p
  }
  # data
  exp3_loc    <- mk_props(df_exp3_woodstone_long, location_code, tank)
  exp3_round  <- mk_props(df_exp3_woodstone_long, location_code, round)
  exp3_loc3   <- mk_props(df_exp3_woodstone_long, reef_type,     tank)
  exp3_round3 <- mk_props(df_exp3_woodstone_long, reef_type,     round)
  # plots
  p_exp3_loc    <- mk_bar(exp3_loc,   location_code, tank,  "Location by tank")
  p_exp3_round  <- mk_bar(exp3_round, location_code, round, "Location by round")
  p_exp3_loc3   <- mk_bar(exp3_loc3,  reef_type,     tank,  "Reeftype by tank",  reef = TRUE)
  p_exp3_round3 <- mk_bar(exp3_round3,reef_type,     round, "Reeftype by round", reef = TRUE)

loc_round_plots_exp3 <<- p_exp3_loc + p_exp3_loc3 + p_exp3_round + p_exp3_round3 +
    patchwork::plot_layout(ncol = 4)
}))

loc_round_plots_exp3

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

(`geom_col()`).

(`geom_text()`).

(`geom_col()`).

(`geom_text()`).

...

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

### Exp 3 Statistic

In [ ]:
# RQ1: Is there a statistical difference between reef type in L1 between the size classes while controlling for group_id, tank and round?
# H0:
# H1: 

# RQ2: Is there a statistical difference between reef type between L0 vs L1? 
# H0:
# H1: 

df3 <- df_exp3_woodstone_long %>%
  mutate(
    reef_type      = factor(reef_type),
    stage          = factor(stage, levels = c("L0", "L1")),
    sex            = factor(sex), # all male
    size_class     = factor(size_class, levels = c("S", "M", "L")),
    location_code  = factor(location_code),
    tank           = factor(tank),
    round          = factor(round),
    group_id       = factor(group_id),
    koura_id       = factor(koura_id)
  ) %>%
  filter(!is.na(reef_type))

## Set FlatStone as reference so coefficients are for WoodSplit vs FlatStone
df3 <- df3 %>% mutate(reef_type = relevel(reef_type, ref = "FlatStone"))

m_exp3 <- glmer(
  reef_type ~ stage + size_class + (1 | group_id) + (1 | tank) + (1 | round),
  data    = df3,
  family  = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa")
)

boundary (singular) fit: see help('isSingular')

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: reef_type ~ stage + size_class + (1 | group_id) + (1 | tank) +  
    (1 | round)
   Data: df3
Control: glmerControl(optimizer = "bobyqa")

      AIC       BIC    logLik -2*log(L)  df.resid 
    274.6     298.1    -130.3     260.6       205 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-1.5084 -0.7957 -0.4794  0.9525  2.1279 

Random effects:
 Groups   Name        Variance Std.Dev.
 group_id (Intercept) 0.03658  0.1912  
 tank     (Intercept) 0.00000  0.0000  
 round    (Intercept) 0.08546  0.2923  
Number of obs: 212, groups:  group_id, 6; tank, 6; round, 3

Fixed effects:
            Estimate Std. Error z value Pr(>|z|)    
(Intercept)   0.6008     0.3473   1.730 0.083646 .  
stageL1      -1.0413     0.3023  -3.444 0.000573 ***
size_classM  -0.6699     0.3584  -1.869 0.061615 .  
size_classL  -1.1195     0.3716  -3.013 0.002586 ** 
---
Sig

## Exp 4 Bricks

### Exp 4 I

In [ ]:
exp4_reef <- df_exp4_brick_long %>%
  dplyr::filter(stage %in% c("L1"), !is.na(reef_type)) %>%
  dplyr::count(stage, reef_type, name = "n") %>%
  dplyr::group_by(stage) %>%
  dplyr::mutate(N = sum(n), prop = n / N) %>%
  dplyr::ungroup() %>%
  relevel_reef4()

# By SEX
exp4_reef_sex <- df_exp4_brick_long %>%
  dplyr::filter(stage %in% c("L1")) %>%
  dplyr::count(stage, reef_type, sex, name = "n") %>%
  dplyr::group_by(stage, sex) %>%
  dplyr::mutate(N = sum(n), prop = n / N) %>%
  dplyr::ungroup() %>%
  relevel_reef4() %>%
  dplyr::mutate(panel = dplyr::recode(sex, F = "Female", M = "Male"))

# By SIZE
exp4_reef_size <- df_exp4_brick_long %>%
  dplyr::filter(stage %in% c("L1")) %>%
  dplyr::count(stage, reef_type, size_class, name = "n") %>%
  dplyr::group_by(stage, size_class) %>%
  dplyr::mutate(N = sum(n), prop = n / N) %>%
  dplyr::ungroup() %>%
  relevel_reef4()

# Labels (facets)
PANEL_LEVELS4  <- c("Combined","M","F")
lab_panel4     <- c(Combined = "Combined", M = "Male", F = "Female")

PANEL_LEVELS5 <- c("Combined","S","M","L")
lab_panel5    <- c(Combined = "Combined", S = "Small", M = "Medium", L = "Large")

# ---- SEX ----
sex_n <- exp4_reef_sex %>%
  group_by(sex) %>%
  summarise(n_total = sum(n), .groups = "drop")

# Build named vector for labeller
lab_panel_n_sex <- setNames(paste0(c("Combined", sex_n$sex),c(" (combined)",paste0(" (n=", sex_n$n_total, ")"))),c("Combined", sex_n$sex))

# Rename F/M to Female/Male with n
lab_panel_n_sex <- c( Combined = "Combined",  M = paste0("Male"), F = paste0("Female"))  

# ---- SIZE ----
size_n <- exp4_reef_size %>%
  group_by(size_class) %>%
  summarise(n_total = sum(n), .groups = "drop")

lab_panel_n_size <- c(  Combined = "Combined",S = paste0("Small"),M = paste0("Medium"),L = paste0("Large"))

# ---------- SEX analysis ----------
# Build combined counts
exp4_sex_top <- exp4_reef %>%
  dplyr::mutate(panel = factor("Combined", levels = PANEL_LEVELS4))

exp4_sex_bot <- exp4_reef_sex %>%
  dplyr::mutate(panel = factor(as.character(sex), levels = PANEL_LEVELS4))

exp4_sex_all <- dplyr::bind_rows(exp4_sex_top, exp4_sex_bot)

# Plot
Exp4_sex_p <- exp4_sex_all %>%
  relevel_reef4() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage,
             labeller = labeller(
               stage = lab_stage,
               panel = lab_panel_n_sex )) +
  labs(x = "Reef type", y = "Proportion")

ggplot2::ggsave(filename = file.path(out_dir, "Exp4_sex_p.png"),plot = Exp4_sex_p,width = 8, height = 8, dpi = 300)

# ---------- SIZE analysis (standalone) ----------

# Build combined counts
exp4_size_top <- exp4_reef %>%
  dplyr::mutate(panel = factor("Combined", levels = PANEL_LEVELS5))

exp4_size_bot <- exp4_reef_size %>%
  dplyr::mutate(panel = factor(as.character(size_class), levels = PANEL_LEVELS5))

exp4_size_all <- dplyr::bind_rows(exp4_size_top, exp4_size_bot)

# Plot
Exp4_size_p <- exp4_size_all %>%
  relevel_reef4() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage,
             labeller = labeller(
               stage = lab_stage,
               panel = lab_panel_n_size)) +
  labs(x = "Reef type", y = "Proportion")

ggplot2::ggsave(filename = file.path(out_dir, "Exp4_size_p.png"), plot = Exp4_size_p, width = 8, height = 8, dpi = 300)

Exp4_size_sex_p <- Exp4_sex_p + Exp4_size_p 

ggplot2::ggsave(filename = file.path(out_dir, "Exp4_size_sex_p.png"), plot = Exp4_size_sex_p, width = 8, height = 8, dpi = 300)

Exp4_size_sex_p

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

### Exp 4 II

In [ ]:
df_exp4_brick_long %>%
  group_by(group_id, size_class, sex, reef_type) %>%
  summarise(n = n(), .groups = "drop") %>%
  arrange(group_id, size_class)

# A tibble: 20 × 5
   group_id size_class sex   reef_type           n
      <dbl> <chr>      <chr> <fct>           <int>
 1        7 L          F     BrickArtificial     2
 2        7 L          F     BrickWood           2
 3        7 M          F     BrickArtificial    15
 4        7 M          F     BrickWood           9
 5        7 M          F     TankWall            4
 6        7 S          F     BrickWood           8
 7        8 M          F     BrickArtificial    11
 8        8 M          F     BrickWood          15
 9        8 M          F     TankWall            2
10        8 M          M     BrickArtificial     2
11        8 M          M     BrickWood           2
12        8 S          F     BrickArtificial     2
13        8 S          F     BrickWood           2
14        8 S          M     BrickArtificial     2
15        8 S          M     BrickWood           2
16        9 M          M     BrickArtificial    15
17        9 M          M     BrickWood          17
18        9 

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBound

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning 

### Exp 4 Statistic

In [ ]:
# RQ1: Is there a statistical difference between reef type in L1 between the size classes while controlling for group_id, tank and round?
# H0:
# H1: 

# RQ2: 
# H0:
# H1: 

df4 <- df_exp4_brick_long %>%
  mutate(
    reef_type      = factor(reef_type),
    stage          = factor(stage, levels = c("L0", "L1")),
    sex            = factor(sex),
    size_class     = factor(size_class, levels = c("S", "M", "L")),
    location_code  = factor(location_code),
    tank           = factor(tank),
    round          = factor(round),
    group_id       = factor(group_id),
    koura_id       = factor(koura_id)) %>%
  filter(!is.na(reef_type))

## Set BrickArtificial as reference so coefficients are for BrickWood vs BrickArtificial
df4 <- df4 %>% mutate(reef_type = relevel(reef_type, ref = "BrickArtificial"))

m_exp4 <- glmer(
  reef_type ~ location_code + sex + size_class + (1 | group_id) + (1 | koura_id) + (1 | tank) + (1 | round),
  data    = df4,
  family  = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa"))

boundary (singular) fit: see help('isSingular')

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: reef_type ~ location_code + sex + size_class + (1 | group_id) +  
    (1 | koura_id) + (1 | tank) + (1 | round)
   Data: df4
Control: glmerControl(optimizer = "bobyqa")

      AIC       BIC    logLik -2*log(L)  df.resid 
    220.7     257.6     -98.4     196.7       148 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-1.7245 -0.9283  0.0001  0.9122  1.1568 

Random effects:
 Groups   Name        Variance  Std.Dev. 
 koura_id (Intercept) 1.019e-14 1.009e-07
 round    (Intercept) 0.000e+00 0.000e+00
 tank     (Intercept) 0.000e+00 0.000e+00
 group_id (Intercept) 0.000e+00 0.000e+00
Number of obs: 160, groups:  koura_id, 40; round, 4; tank, 4; group_id, 4

Fixed effects:
                 Estimate Std. Error z value Pr(>|z|)  
(Intercept)       1.40759    0.69317   2.031   0.0423 *
location_codeB   -0.34486    0.48257  -0.715   0.4748  
location_

## Waterquality

In [ ]:
wq_clean <- df_wq %>%
  mutate(value = suppressWarnings(as.numeric(value)),tank  = factor(tank),param_label = ifelse(is.na(unit) | unit %in% c("", "NA"), parameter_abb, paste(parameter_abb, unit))) %>%
  filter(!is.na(tank_use), !is.na(value),data_time >= as.POSIXct("2025-09-01"),tank_use == "Experiment")

wq_probe <- wq_clean %>%
  transmute(datetime = data_time,tank,param_label,value)

# --- Logger data (df_loggers) -> give unique param_label names ---
wq_logger <- df_loggers %>%
  rename(datetime = data_time) %>%
  mutate(tank     = factor(tank),
    datetime = lubridate::ymd_hms(datetime, tz = "Pacific/Auckland")) %>%
  pivot_longer(c(temperature, light), names_to = "param", values_to = "value") %>%
  mutate(
    param_label = case_when(
      param == "temperature" ~ "Temp (°C) - Logger",
      param == "light"       ~ "Light (lux) - Logger",
      TRUE                   ~ param),
    date  = as.Date(datetime),
    time  = format(datetime, "%H:%M:%S"),
    year  = year(datetime),
    month = month(datetime, label = TRUE),
    day   = day(datetime)) %>%
  filter(
    datetime >= as.POSIXct("2025-09-01", tz = "Pacific/Auckland"),
    (param_label != "Light (lux) - Logger" | value <= 1000),
    (param_label != "Temp (°C) - Logger"   | value <= 20) ) %>%
  transmute(datetime, date, time, year, month, day, tank, param_label, value)

# --- Combine into ONE df, keep labels distinct ---
PARAM_LEVELS <- c("Temp (°C)", "Temp (°C) - Logger","Light (lux) - Logger",
                  "Press (mbars)","DO (%)","DO (mg/L)","SPC (µS/cm)","Cond (µS/cm)","SAL (ppt)","pH")

wq_all <- bind_rows(wq_probe, wq_logger) %>%
  mutate(param_label = factor(param_label, levels = PARAM_LEVELS),tank = factor(tank)) %>%
  tidyr::drop_na(value)

# --- Plot from the single df ---
wq_all_plot <- ggplot(wq_all, aes(datetime, value, color = tank, group = tank)) +
  geom_line() +
  facet_wrap(~ param_label, scales = "free_y", ncol = 2)

wq_all_plot

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database

# Discussion

## Conclusion

# Acknowledgements

# References

# Supplementary information

## Length-Weight of koura

In [ ]:
summarise_size_weight <- function(df) {
  df %>%
    summarise(
      n_size          = sum(!is.na(size_mm1)),
      n_weight        = sum(!is.na(weight_g1)),
      size_mm1_mean   = mean(size_mm1, na.rm = TRUE),
      size_mm1_sd     = sd(size_mm1, na.rm = TRUE),
      size_mm1_range  = paste0(min(size_mm1, na.rm = TRUE), "–", max(size_mm1, na.rm = TRUE)),
      weight_g1_mean  = mean(weight_g1, na.rm = TRUE),
      weight_g1_sd    = sd(weight_g1, na.rm = TRUE),
      weight_g1_range = paste0(min(weight_g1, na.rm = TRUE), "–", max(weight_g1, na.rm = TRUE))
    )
}

summarise_size_weight(df_exp1_single_raw)

# A tibble: 1 × 8
  n_size n_weight size_mm1_mean size_mm1_sd size_mm1_range weight_g1_mean
   <int>    <int>         <dbl>       <dbl> <chr>                   <dbl>
1     30       30          26.5        2.13 22.61–29.97              12.2
# ℹ 2 more variables: weight_g1_sd <dbl>, weight_g1_range <chr>

# A tibble: 1 × 8
  n_size n_weight size_mm1_mean size_mm1_sd size_mm1_range weight_g1_mean
   <int>    <int>         <dbl>       <dbl> <chr>                   <dbl>
1    180      180          27.3        7.57 15.77–43.34              15.8
# ℹ 2 more variables: weight_g1_sd <dbl>, weight_g1_range <chr>

# A tibble: 1 × 8
  n_size n_weight size_mm1_mean size_mm1_sd size_mm1_range weight_g1_mean
   <int>    <int>         <dbl>       <dbl> <chr>                   <dbl>
1    108      108          27.4        7.56 15.77–43.34              15.9
# ℹ 2 more variables: weight_g1_sd <dbl>, weight_g1_range <chr>

# A tibble: 1 × 8
  n_size n_weight size_mm1_mean size_mm1_sd size_mm1_range weight_g1_mean
   <int>    <int>         <dbl>       <dbl> <chr>                   <dbl>
1    160      160          26.1        2.58 20.21–30.81              11.9
# ℹ 2 more variables: weight_g1_sd <dbl>, weight_g1_range <chr>

Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database
Warning in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, : font
family not found in Windows font database

Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database
Warning in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
font family not found in Windows font database

```` markdown
---
title: ""
subtitle: ""

author:
  - name: Olivier V. Raven
    orcid: 0009-0008-6184-6831
    affiliations:
      - name: University of Waikato
        department: Te Aka Mātuatua – School of Science
        address: Private Bag 3105, Hamilton 3240
        country: New Zealand
    email: olivier.raven@icloud.com; or81@students.waikato.ac.nz
    corresponding: true
    
  - name: Calum MacNeil
    orcid: 0000-0003-0402-9292
    affiliations:
      - name: The Cawthron Institute
        address: 98 Halifax Street East, Nelson 7010, Private Bag 2, Nelson 7042
        country: New Zealand
              
  - name: Robin Holmes
    orcid: 0000-0002-8737-2717
    affiliations:
      - name: The Cawthron Institute
        address: 98 Halifax Street East, Nelson 7010, Private Bag 2, Nelson 7042
        country: New Zealand
        
  - name: Ian A. Kusabs
    orcid: 0000-0002-1473-7447
    affiliations:
      - name: Ian Kusabs and Associates Limited
        address: 21 Summit Road, Rotorua 3076
        country: New Zealand
        
  - name: Francis J. Burdon
    orcid: 0000-0002-5398-4993
    affiliations:
      - name: University of Waikato
        department: Te Aka Mātuatua – School of Science
        address: Private Bag 3105, Hamilton 3240
        country: New Zealand
        
  - name: Deniz Özkundakci
    orcid: 0000-0002-5442-4576
    affiliations:
      - name: University of Waikato
        department: Te Aka Mātuatua – School of Science
        address: Private Bag 3105, Hamilton 3240
        country: New Zealand

date: "..."

abstract: |
 
keywords: koura

funding: "This research was supported by the Fish Futures programme funded through a Ministry of Business, Innovation and Employment grant (CAWX2101)"

citation:
  container-title: "..."
  
format:
  html:
    theme: cosmo
---

# Introduction
# Materials and Methods
# Results
## Setup
quarto-executable-code-5450563D

```r
#| label: Setup and load-data
#| include: false
packages <- c("XNomial","brms","nnet","lme4","multcompView","patchwork","janitor","lubridate","stringr","tidyverse","dplyr","ggplot2","readxl","writexl","readr")

quiet_load <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    suppressWarnings(suppressMessages(install.packages(pkg, dependencies = TRUE)))}
  suppressPackageStartupMessages(require(pkg, character.only = TRUE, quietly = TRUE))
  invisible(TRUE)}

invisible(lapply(packages, quiet_load))

# Set paths
exc_file_dir    <- "data/raw/Mesocosm_experiment.xlsx"
raw_data_dir    <- "data/raw"
der_data_dir    <- "data/derived"
out_dir         <- "outputs"

read_sheet <- function(sheet) {readxl::read_excel(exc_file_dir, sheet = sheet)}
write_out <- function(df, name) {write.csv(df, file.path(der_data_dir, name), row.names = FALSE)}

# Read data
df_koura               <- read_sheet("Koura")
df_setup               <- read_sheet("Tank_setup")
df_exp1_single_raw     <- read_sheet("Reef_single")
df_exp2_group_raw      <- read_sheet("Reef_group")
df_exp3_woodstone_raw  <- read_sheet("Stone_wood")
df_exp4_brick_raw      <- read_sheet("Bricks")
df_wq                  <- read_sheet("WQ")
df_loggers             <- read_sheet("Loggers")

# Save as csv to derived
write_out(df_koura,              "df_koura.csv")
write_out(df_setup,              "df_setup.csv")
write_out(df_exp1_single_raw,    "df_exp1_single_raw.csv")
write_out(df_exp2_group_raw,     "df_exp2_group_raw.csv")
write_out(df_exp3_woodstone_raw, "df_exp3_woodstone_raw.csv")
write_out(df_exp4_brick_raw,     "df_exp4_brick_raw.csv")
write_out(df_wq,                 "df_wq.csv")
write_out(df_loggers,            "df_loggers.csv")

# set preferred orders
REEF_LEVELS  <- c("SingleStone","FlatStone","StonePile","WoodLog","TankWall")
REEF_LEVELS3 <- c("FlatStone","WoodSplit","TankWall", "NA")
REEF_LEVELS4 <- c("BrickArtificial","BrickWood","TankWall")
LOC_LEVELS   <- c("A", "B", "C", "D", "E")
LOC_LEVELS3  <- c("A", "B", "E")

relevel_reef  <- function(df) df %>% mutate(reef_type = factor(as.character(reef_type),levels = REEF_LEVELS))
relevel_reef3 <- function(df) df %>% mutate(reef_type = factor(as.character(reef_type),levels = REEF_LEVELS3))
relevel_reef4 <- function(df) df %>% mutate(reef_type = factor(as.character(reef_type),levels = REEF_LEVELS4))

# Set plot theme
base_theme_bw <- theme_classic() +
  theme(text     = element_text(family = "Arial", size = 11),
    axis.title   = element_text(face = "plain"),
    axis.text    = element_text(face = "plain"),
    plot.title   = element_text(face = "plain"),
    strip.text   = element_text(face = "plain"),
    panel.border = element_rect(colour = "black", fill = NA, linewidth = 0.5))
theme_set(base_theme_bw)
```


## Make DF's into long format
quarto-executable-code-5450563D

```r
#| label: Long format

df_exp1_single_long <- df_exp1_single_raw %>%
  mutate(location_l0 = l0, location_l1 = l1) %>%
  pivot_longer(cols = matches("_(l0|l1)$"),
    names_to = c(".value", "stage"),
    names_pattern = "^(.*)_(l[01])$") %>%
  mutate(stage         = toupper(stage),
    location_code = factor(toupper(location), levels = LOC_LEVELS),
    reef_type     = factor(reef, levels = REEF_LEVELS)) %>%
  select(tank, round, group_id, koura_id, size_class, sex, size_mm1, weight_g1, date, stage, t_leave_s, t_l0_s, location_code, reef_type,legs1, claws1, antenna1, size_mm2, weight_g2, legs2, claws2, antenna2,notes_exp, notes_koura)

write_out(df_exp1_single_long, "df_exp1_single_long.csv")

# EXP 2 – Group reefs (long)
df_exp2_group_long <- df_exp2_group_raw %>%
  mutate(location_l0 = l0, location_l1 = l1) %>%
  pivot_longer(cols = c(location_l0, location_l1,
      reef_l0, reef_l1,legs_l0, claws_l0, antenna_l0,legs_l1, claws_l1, antenna_l1), names_to = c(".value", "stage"), names_pattern = "^(.*)_(l[01])$") %>%
  mutate(stage          = toupper(stage),
    location_code  = factor(toupper(location), levels = LOC_LEVELS),
    reef_type      = factor(reef, levels = REEF_LEVELS),
    size_mm_stage  = if_else(stage == "L0", size_mm1, size_mm2, NA_real_),
    weight_g_stage = if_else(stage == "L0", weight_g1, weight_g2, NA_real_)) %>%
  select(tank, round, group_id, koura_id, size_class, sex, date,stage, t_leave_s, t_l0_s, location_code, reef_type,legs, claws, antenna, size_mm_stage, weight_g_stage,size_mm1, weight_g1, legs1, claws1, antenna1,size_mm2, weight_g2, legs2, claws2, antenna2,notes_exp, notes_koura)

write_out(df_exp2_group_long, "df_exp2_group_long.csv")


# EXP 3 – Stone / wood (long)
df_exp3_woodstone_long <- df_exp3_woodstone_raw %>%
  mutate(location_l0 = l0, location_l1 = l1) %>%
  pivot_longer(cols = c(location_l0, location_l1, reef_l0, reef_l1),
    names_to = c(".value", "stage"),
    names_pattern = "^(.*)_(l[01])$") %>%
  mutate(stage         = toupper(stage),
    location_code = factor(toupper(location), levels = LOC_LEVELS3),
    reef_type     = factor(reef, levels = REEF_LEVELS3)) %>%
  select(tank, round, group_id, koura_id, size_class, sex, date, stage, location_code, reef_type, size_mm1, weight_g1, legs1, claws1, antenna1, size_mm2, weight_g2, legs2, claws2, antenna2, notes_exp, notes_koura)

write_out(df_exp3_woodstone_long, "df_exp3_woodstone_long.csv")


# EXP 4 – Bricks (long)
df_exp4_brick_long <- df_exp4_brick_raw %>%
  mutate(location_l1 = l1) %>%
  pivot_longer(cols = c(location_l1, reef_l1),
    names_to = c(".value", "stage"),
    names_pattern = "^(.*)_(l[01])$") %>%
  mutate(stage         = toupper(stage),
    location_code = factor(toupper(location), levels = LOC_LEVELS),
    reef_type     = factor(reef, levels = REEF_LEVELS4)) %>%
  select(tank, round, group_id, koura_id, size_class, sex, date, stage, location_code, reef_type,size_mm1, weight_g1, legs1, claws1, antenna1,size_mm2, weight_g2, legs2, claws2, antenna2,notes_exp, notes_koura)

write_out(df_exp4_brick_long, "df_exp4_brick_long.csv")
```

## Exp 1 Single koura
### Exp 1 I
quarto-executable-code-5450563D

```r
#| label: Exp 1
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

exp1_reef_combined <- df_exp1_single_long %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, name = "n") %>%
  group_by(stage) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef() %>%
  mutate(panel = "Combined")

exp1_reef_bysex <- df_exp1_single_long %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, sex, name = "n") %>%
  group_by(stage, sex) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef() %>%
  mutate(panel = dplyr::recode(sex, F = "Female", M = "Male")) %>%
  select(-sex)

exp1_reef_all <- dplyr::bind_rows(exp1_reef_combined, exp1_reef_bysex)

lab_stage <- c(L0 = "Initial location (L0)", L1 = "Next-day location (L1)")
lab_panel1 <- c(Combined = "Combined", Female = "Female", Male = "Male")

# One plot ADD: n values above each bar and in 
Exp1_reef_p <- exp1_reef_all %>%
  relevel_reef() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage, labeller = labeller(stage = lab_stage, panel = lab_panel1)) +
  labs(x = "Reef type", y = "Proportion")

ggsave(Exp1_reef_p, file = file.path(out_dir, "Exp1_reef_p.png"), width = 8, height = 6, dpi = 300)

Exp1_reef_p
```

### Exp 1 II
quarto-executable-code-5450563D

```r
#| label: Exp 1 II
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

invisible(capture.output({
  # proportions by x var and facet var (per stage)
  mk_props <- function(data, xvar, facetvar) {
    data %>%
      dplyr::filter(stage %in% c("L0","L1")) %>%
      dplyr::count(stage, {{ xvar }}, {{ facetvar }}, name = "n") %>%
      dplyr::group_by(stage, {{ facetvar }}) %>%
      dplyr::mutate(N = sum(n), prop = n / N) %>%
      dplyr::ungroup()
  }

  # bar plot; rows = facetvar, cols = stage; fixed y-scale across all plots
  mk_bar <- function(df, xvar, facetvar, title, reef = FALSE) {
    p <- ggplot(df, aes({{ xvar }}, prop)) +
      geom_col(color = "black") +
      geom_text(aes(label = n), vjust = 1.3, colour = "white") +
      facet_grid(rows = vars({{ facetvar }}), cols = vars(stage)) +
      labs(title = title) +
      scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.05)))
    if (reef) {
      p <- p +
        scale_x_discrete(limits = REEF_LEVELS, drop = FALSE) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))
    }
    p
  }

  # data
  exp1_loc    <- mk_props(df_exp1_single_long, location_code, tank)
  exp1_round  <- mk_props(df_exp1_single_long, location_code, round)
  exp1_loc2   <- mk_props(df_exp1_single_long, reef_type,     tank)
  exp1_round2 <- mk_props(df_exp1_single_long, reef_type,     round)

  # plots
  p_exp1_loc    <- mk_bar(exp1_loc,   location_code, tank,  "Location by tank")
  p_exp1_round  <- mk_bar(exp1_round, location_code, round, "Location by round")
  p_exp1_loc2   <- mk_bar(exp1_loc2,  reef_type,     tank,  "Reeftype by tank",  reef = TRUE)
  p_exp1_round2 <- mk_bar(exp1_round2,reef_type,     round, "Reeftype by round", reef = TRUE)

  loc_round_plots_exp1 <<- p_exp1_loc + p_exp1_loc2 + p_exp1_round + p_exp1_round2 +
    patchwork::plot_layout(ncol = 4)
}))
loc_round_plots_exp1

```

### Exp 1 III
quarto-executable-code-5450563D

```r
#| label: Exp 1 III
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

exp1_pairs <- df_exp1_single_long %>%
  dplyr::filter(stage %in% c("L0","L1")) %>%
  dplyr::select(koura_id, tank, round, size_class, sex, stage,location_code, reef_type, t_leave_s, t_l0_s) %>%
  dplyr::mutate(
    location_code = forcats::fct_drop(factor(location_code, levels = LOC_LEVELS)),
    reef_type     = forcats::fct_drop(factor(reef_type,     levels = REEF_LEVELS)),
    t_leave_s     = suppressWarnings(as.numeric(t_leave_s)),
    t_l0_s        = suppressWarnings(as.numeric(t_l0_s))) %>%
  tidyr::pivot_wider(names_from = stage,values_from = c(location_code, reef_type, t_leave_s, t_l0_s),names_sep = "_")

activity_df <- exp1_pairs %>%
  dplyr::transmute(koura_id, tank, round, size_class, sex, time_to_leave = t_leave_s_L0, time_at_L0 = t_l0_s_L0) 

activity_long <- activity_df %>%
  mutate(
    time_to_leave = suppressWarnings(as.numeric(time_to_leave)),time_at_L0    = suppressWarnings(as.numeric(time_at_L0))) %>%
  pivot_longer(c(time_to_leave, time_at_L0), names_to = "metric", values_to = "time") %>%
  filter(!is.na(time), time >= 0)

lab_metric <- c(time_to_leave = "Time to leave start (t_leave)",
                time_at_L0    = "Time to reach initial hide (t_L0)")

act_exp1_p <- ggplot(activity_long, aes(time, col=sex)) +
  stat_ecdf(geom = "step") +
  facet_grid( ~ metric, labeller = labeller(metric = lab_metric)) +
  labs(x = "Time (s)", y = "Proportion completed by time")

act_exp1_p

```

### Exp 1 Statistic
quarto-executable-code-5450563D

```r
#| label: Statistics Exp 1
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

# RQ1: 
# H0:
# H1: 

# RQ2: Is there a statistical difference between reef type between L0 vs L1? 
# H0:
# H1: 


df1 <- df_exp1_single_long %>%
  mutate(
    reef_type      = factor(reef_type),
    stage          = factor(stage, levels = c("L0", "L1")),
    sex            = factor(sex),
    size_class     = factor(size_class),
    location_code  = factor(location_code),
    tank           = factor(tank),
    round          = factor(round),
    koura_id       = factor(koura_id),
    size_mm1       = as.numeric(size_mm1),
    weight_g1      = as.numeric(weight_g1)) %>%
  filter(!is.na(reef_type))

# most preferred reef at L1
df1_L1 <- df1 %>% filter(stage == "L1")

tab_L1 <- table(df1_L1$reef_type)
tab_L1_nz <- tab_L1[tab_L1 > 0]        
XNomial::xmulti(obs  = as.numeric(tab_L1_nz), expr = rep(sum(tab_L1_nz) / length(tab_L1_nz), length(tab_L1_nz)))



# Test between L0 and L1
m_exp1 <- nnet::multinom(reef_type ~ stage + sex + size_mm1, data = df1)
m_exp1. <- nnet::multinom(reef_type ~ sex + size_mm1, data = df1)
summary(m_exp1)
summary(m_exp1.)
confint(m_exp1)
anova(m_exp1, m_exp1., test = "Chisq")
```

## Exp 2 Group koura
### Exp 2 I
quarto-executable-code-5450563D

```r
#| label: Exp 2
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

exp2_reef_combined <- df_exp2_group_long %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, name = "n") %>%
  group_by(stage) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef() %>%
  mutate(panel = "Combined")

# By size class
exp2_reef_bysize <- df_exp2_group_long %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, size_class, name = "n") %>%
  group_by(stage, size_class) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef() %>%
  mutate(panel = dplyr::recode(as.character(size_class), S = "Small", M = "Medium", L = "Large")) %>%
  select(-size_class)

# Bind for one facetted plot
exp2_reef_all <- dplyr::bind_rows(exp2_reef_combined, exp2_reef_bysize)

# Labels 
lab_panel2 <- c(Combined = "Combined", Small = "Small", Medium = "Medium", Large = "Large")

# One plot
Exp2_reef_p <- exp2_reef_all %>%
  relevel_reef() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage, labeller = labeller(stage = lab_stage, panel = lab_panel2)) +
  labs(x = "Reef type", y = "Proportion")

ggsave(Exp2_reef_p, file = file.path(out_dir, "Exp2_reef_p.png"), width = 8, height = 8, dpi = 300)

Exp2_reef_p
```

### Exp 2 II
quarto-executable-code-5450563D

```r
#| label: Exp 2 II
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

invisible(capture.output({

  # proportions by x var and facet var (per stage)
  mk_props <- function(data, xvar, facetvar) {
    data %>%
      dplyr::filter(stage %in% c("L0","L1")) %>%
      dplyr::count(stage, {{ xvar }}, {{ facetvar }}, name = "n") %>%
      dplyr::group_by(stage, {{ facetvar }}) %>%
      dplyr::mutate(N = sum(n), prop = n / N) %>%
      dplyr::ungroup()
  }

  # bar plot; uses facet_grid for rows=facetvar, cols=stage
  mk_bar <- function(df, xvar, facetvar, title, reef = FALSE) {
    p <- ggplot(df, aes({{ xvar }}, prop)) +
      geom_col(color = "black") +
      geom_text(aes(label = n), vjust = 1.3, colour = "white") +
      facet_grid(rows = vars({{ facetvar }}), cols = vars(stage)) +
      labs(title = title)+
      scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.05)))
    if (reef) {
      p <- p +
        scale_x_discrete(limits = REEF_LEVELS, drop = FALSE) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))
    }
    p
  }

  # data
  exp2_loc    <- mk_props(df_exp2_group_long, location_code, tank)
  exp2_round  <- mk_props(df_exp2_group_long, location_code, round)
  exp2_loc2   <- mk_props(df_exp2_group_long, reef_type,     tank)
  exp2_round2 <- mk_props(df_exp2_group_long, reef_type,     round)

  # plots
  p_exp2_loc    <- mk_bar(exp2_loc,   location_code, tank,  "Location by tank")
  p_exp2_round  <- mk_bar(exp2_round, location_code, round, "Location by round")
  p_exp2_loc2   <- mk_bar(exp2_loc2,  reef_type,     tank,  "Reeftype by tank",  reef = TRUE)
  p_exp2_round2 <- mk_bar(exp2_round2,reef_type,     round, "Reeftype by round", reef = TRUE)

  loc_round_plots_exp2 <<- p_exp2_loc + p_exp2_loc2 + p_exp2_round + p_exp2_round2 +
    patchwork::plot_layout(ncol = 4)
}))
loc_round_plots_exp2

```

### Exp 2 III
quarto-executable-code-5450563D

```r
#| label: Exp 2 III
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

exp2_reef_koura <- df_exp2_group_long %>%
  mutate(stage = toupper(stage)) %>%
  filter(stage %in% c("L0","L1")) %>%
  count(stage, reef_type, koura_id, name = "n") %>%
  group_by(stage, koura_id) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>%
  relevel_reef()

p_exp2_reef_koura <- exp2_reef_koura %>%
  relevel_reef() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  facet_wrap(koura_id ~ stage, nrow = 6) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

p_exp2_reef_koura


exp2_use <- df_exp2_group_long %>% 
  filter(stage %in% c("L0","L1"))

counts <- exp2_use %>%
  filter(!is.na(reef_type), !is.na(size_class)) %>%
  mutate(
    tank = factor(tank),
    round = factor(round),
    stage = factor(stage, levels = c("L0","L1")),
    reef_type  = fct_relevel(reef_type,"SingleStone","FlatStone","StonePile","WoodLog","TankWall"),
    size_class = factor(size_class)) %>%
  count(stage, round, tank, reef_type, size_class, name = "number_koura") %>%
  complete(stage, round, tank, reef_type, size_class, fill = list(number_koura = 0))

p_stack <- ggplot(counts, aes(reef_type, number_koura, fill = size_class)) +
  geom_col() +
  facet_grid(round ~ tank+stage  , scales = "free_x")+
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

p_stack
```

### Exp 2 Statistic
quarto-executable-code-5450563D

```r
#| label: Statistics Exp 2
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

# RQ1: Is there a statistical difference between reef type in L1 between the size classes while controlling for group_id, tank and round?
# H0:
# H1: 

# RQ2: Is there a statistical difference between reef type between L0 vs L1? 
# H0:
# H1: 

df2 <- df_exp2_group_long %>%
  mutate(
    reef_type      = factor(reef_type),
    stage          = factor(stage, levels = c("L0", "L1")),
    sex            = factor(sex), # exclude as all male
    size_class     = factor(size_class, levels = c("S", "M", "L")),
    location_code  = factor(location_code),
    tank           = factor(tank),
    round          = factor(round),
    group_id       = factor(group_id), # each group of 6 koura used 5 times
    koura_id       = factor(koura_id)) %>%
  filter(!is.na(reef_type))

form_exp2 <- reef_type ~ stage + size_class + (1 | group_id) + (1 | tank) + (1 | round)

#m_exp2 <- brm(formula = form_exp2, data = df2, family = categorical(link = "logit"), chains= 4, cores = 4,  iter = 4000, seed = 1)

#summary(m_exp2)
```

## Exp 3 Wood vs stone
### Exp 3 I
quarto-executable-code-5450563D

```r
#| label: Exp 3
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

exp3_reef <- df_exp3_woodstone_long %>%
  filter(stage %in% c("L0","L1"), !is.na(reef_type)) %>%
  count(stage, reef_type, name = "n") %>%
  group_by(stage) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>% relevel_reef3()

exp3_reef_size <- df_exp3_woodstone_long %>%
  filter(stage %in% c("L0","L1"), !is.na(reef_type)) %>%
  count(stage, reef_type, size_class, name = "n") %>%
  group_by(stage, size_class) %>%
  mutate(N = sum(n), prop = n / N) %>%
  ungroup() %>% relevel_reef3()

# Panel labels and order
PANEL_LEVELS <- c("Combined","L","M","S")
lab_panel3    <- c(Combined = "Combined", L = "Large", M = "Medium", S = "Small")

# Build combined counts
exp3_top <- exp3_reef %>%
  mutate(panel = factor("Combined", levels = PANEL_LEVELS))

exp3_bot <- exp3_reef_size %>%
  mutate(panel = factor(as.character(size_class), levels = PANEL_LEVELS))

exp3_all <- bind_rows(exp3_top, exp3_bot)

# One plot with a left strip for every row (Combined, S, M, L)
Exp3_reef_p <- exp3_all %>%
  #dplyr::left_join(exp3_all_L, by = c("stage","reef_type","panel")) %>% relevel_reef3() %>% 
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +  
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage, labeller = labeller(stage = lab_stage, panel = lab_panel3)) +
  labs(x = "Reef type", y = "Proportion")

ggsave(Exp3_reef_p, file = file.path(out_dir, "Exp3_reef_p.png"), width = 8, height = 8, dpi = 300)

Exp3_reef_p

```

### Exp 3 II
quarto-executable-code-5450563D

```r
#| label: Exp 3 II
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

invisible(capture.output({
  # proportions by x var and facet var (per stage)
  mk_props <- function(data, xvar, facetvar) {
    data %>%
      dplyr::filter(stage %in% c("L0","L1")) %>%
      dplyr::count(stage, {{ xvar }}, {{ facetvar }}, name = "n") %>%
      dplyr::group_by(stage, {{ facetvar }}) %>%
      dplyr::mutate(N = sum(n), prop = n / N) %>%
      dplyr::ungroup()
  }

# bar plot; uses facet_grid for rows=facetvar, cols=stage
  mk_bar <- function(df, xvar, facetvar, title, reef = FALSE) {
    p <- ggplot(df, aes({{ xvar }}, prop)) +
      geom_col(color = "black") +
      geom_text(aes(label = n), vjust = 1.3, colour = "white") +
      facet_grid(rows = vars({{ facetvar }}), cols = vars(stage)) +
      labs(title = title)+
      scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.05)))
    if (reef) {
      p <- p +
        scale_x_discrete(limits = REEF_LEVELS3, drop = FALSE) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))
    }
    p
  }
  # data
  exp3_loc    <- mk_props(df_exp3_woodstone_long, location_code, tank)
  exp3_round  <- mk_props(df_exp3_woodstone_long, location_code, round)
  exp3_loc3   <- mk_props(df_exp3_woodstone_long, reef_type,     tank)
  exp3_round3 <- mk_props(df_exp3_woodstone_long, reef_type,     round)
  # plots
  p_exp3_loc    <- mk_bar(exp3_loc,   location_code, tank,  "Location by tank")
  p_exp3_round  <- mk_bar(exp3_round, location_code, round, "Location by round")
  p_exp3_loc3   <- mk_bar(exp3_loc3,  reef_type,     tank,  "Reeftype by tank",  reef = TRUE)
  p_exp3_round3 <- mk_bar(exp3_round3,reef_type,     round, "Reeftype by round", reef = TRUE)

loc_round_plots_exp3 <<- p_exp3_loc + p_exp3_loc3 + p_exp3_round + p_exp3_round3 +
    patchwork::plot_layout(ncol = 4)
}))

loc_round_plots_exp3
```

### Exp 3 Statistic
quarto-executable-code-5450563D

```r
#| label: Statistics Exp 3
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

# RQ1: Is there a statistical difference between reef type in L1 between the size classes while controlling for group_id, tank and round?
# H0:
# H1: 

# RQ2: Is there a statistical difference between reef type between L0 vs L1? 
# H0:
# H1: 

df3 <- df_exp3_woodstone_long %>%
  mutate(
    reef_type      = factor(reef_type),
    stage          = factor(stage, levels = c("L0", "L1")),
    sex            = factor(sex), # all male
    size_class     = factor(size_class, levels = c("S", "M", "L")),
    location_code  = factor(location_code),
    tank           = factor(tank),
    round          = factor(round),
    group_id       = factor(group_id),
    koura_id       = factor(koura_id)
  ) %>%
  filter(!is.na(reef_type))

## Set FlatStone as reference so coefficients are for WoodSplit vs FlatStone
df3 <- df3 %>% mutate(reef_type = relevel(reef_type, ref = "FlatStone"))

m_exp3 <- glmer(
  reef_type ~ stage + size_class + (1 | group_id) + (1 | tank) + (1 | round),
  data    = df3,
  family  = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa")
)

summary(m_exp3)
```

## Exp 4 Bricks
### Exp 4 I
quarto-executable-code-5450563D

```r
#| label: Exp 4 
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

exp4_reef <- df_exp4_brick_long %>%
  dplyr::filter(stage %in% c("L1"), !is.na(reef_type)) %>%
  dplyr::count(stage, reef_type, name = "n") %>%
  dplyr::group_by(stage) %>%
  dplyr::mutate(N = sum(n), prop = n / N) %>%
  dplyr::ungroup() %>%
  relevel_reef4()

# By SEX
exp4_reef_sex <- df_exp4_brick_long %>%
  dplyr::filter(stage %in% c("L1")) %>%
  dplyr::count(stage, reef_type, sex, name = "n") %>%
  dplyr::group_by(stage, sex) %>%
  dplyr::mutate(N = sum(n), prop = n / N) %>%
  dplyr::ungroup() %>%
  relevel_reef4() %>%
  dplyr::mutate(panel = dplyr::recode(sex, F = "Female", M = "Male"))

# By SIZE
exp4_reef_size <- df_exp4_brick_long %>%
  dplyr::filter(stage %in% c("L1")) %>%
  dplyr::count(stage, reef_type, size_class, name = "n") %>%
  dplyr::group_by(stage, size_class) %>%
  dplyr::mutate(N = sum(n), prop = n / N) %>%
  dplyr::ungroup() %>%
  relevel_reef4()

# Labels (facets)
PANEL_LEVELS4  <- c("Combined","M","F")
lab_panel4     <- c(Combined = "Combined", M = "Male", F = "Female")

PANEL_LEVELS5 <- c("Combined","S","M","L")
lab_panel5    <- c(Combined = "Combined", S = "Small", M = "Medium", L = "Large")

# ---- SEX ----
sex_n <- exp4_reef_sex %>%
  group_by(sex) %>%
  summarise(n_total = sum(n), .groups = "drop")

# Build named vector for labeller
lab_panel_n_sex <- setNames(paste0(c("Combined", sex_n$sex),c(" (combined)",paste0(" (n=", sex_n$n_total, ")"))),c("Combined", sex_n$sex))

# Rename F/M to Female/Male with n
lab_panel_n_sex <- c( Combined = "Combined",  M = paste0("Male"), F = paste0("Female"))  

# ---- SIZE ----
size_n <- exp4_reef_size %>%
  group_by(size_class) %>%
  summarise(n_total = sum(n), .groups = "drop")

lab_panel_n_size <- c(  Combined = "Combined",S = paste0("Small"),M = paste0("Medium"),L = paste0("Large"))

# ---------- SEX analysis ----------
# Build combined counts
exp4_sex_top <- exp4_reef %>%
  dplyr::mutate(panel = factor("Combined", levels = PANEL_LEVELS4))

exp4_sex_bot <- exp4_reef_sex %>%
  dplyr::mutate(panel = factor(as.character(sex), levels = PANEL_LEVELS4))

exp4_sex_all <- dplyr::bind_rows(exp4_sex_top, exp4_sex_bot)

# Plot
Exp4_sex_p <- exp4_sex_all %>%
  relevel_reef4() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage,
             labeller = labeller(
               stage = lab_stage,
               panel = lab_panel_n_sex )) +
  labs(x = "Reef type", y = "Proportion")

ggplot2::ggsave(filename = file.path(out_dir, "Exp4_sex_p.png"),plot = Exp4_sex_p,width = 8, height = 8, dpi = 300)

# ---------- SIZE analysis (standalone) ----------

# Build combined counts
exp4_size_top <- exp4_reef %>%
  dplyr::mutate(panel = factor("Combined", levels = PANEL_LEVELS5))

exp4_size_bot <- exp4_reef_size %>%
  dplyr::mutate(panel = factor(as.character(size_class), levels = PANEL_LEVELS5))

exp4_size_all <- dplyr::bind_rows(exp4_size_top, exp4_size_bot)

# Plot
Exp4_size_p <- exp4_size_all %>%
  relevel_reef4() %>%
  ggplot(aes(reef_type, prop)) +
  geom_col(color = "black") +
  geom_text(aes(label = n), vjust = 1.3, colour = "white") +
  facet_grid(panel ~ stage,
             labeller = labeller(
               stage = lab_stage,
               panel = lab_panel_n_size)) +
  labs(x = "Reef type", y = "Proportion")

ggplot2::ggsave(filename = file.path(out_dir, "Exp4_size_p.png"), plot = Exp4_size_p, width = 8, height = 8, dpi = 300)

Exp4_size_sex_p <- Exp4_sex_p + Exp4_size_p 

ggplot2::ggsave(filename = file.path(out_dir, "Exp4_size_sex_p.png"), plot = Exp4_size_sex_p, width = 8, height = 8, dpi = 300)

Exp4_size_sex_p
```

### Exp 4 II
quarto-executable-code-5450563D

```r
#| label: Exp 4 II
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

df_exp4_brick_long %>%
  group_by(group_id, size_class, sex, reef_type) %>%
  summarise(n = n(), .groups = "drop") %>%
  arrange(group_id, size_class)

df_exp4_brick_long %>%
  ggplot(aes(x = factor(group_id), fill = size_class)) +
  geom_bar(position = "fill") +  # use "stack" to show raw counts
  labs(x = "Group ID", y = "Proportion", fill = "Size class") +
  theme_bw()

df_exp4_brick_long %>%
  ggplot(aes(x = factor(group_id), fill = interaction(sex, size_class))) +
  geom_bar(position = "fill") +
  labs(x = "Group ID", y = "Proportion", fill = "Sex × Size class") +
  theme_bw()

invisible(capture.output({
  # proportions by x var and facet var (per stage)
  mk_props <- function(data, xvar, facetvar) {
    data %>%
      dplyr::filter(stage %in% c("L0","L1")) %>%
      dplyr::count(stage, {{ xvar }}, {{ facetvar }}, name = "n") %>%
      dplyr::group_by(stage, {{ facetvar }}) %>%
      dplyr::mutate(N = sum(n), prop = n / N) %>%
      dplyr::ungroup()
  }
  # bar plot; uses facet_grid for rows=facetvar, cols=stage
  mk_bar <- function(df, xvar, facetvar, title, reef = FALSE) {
    p <- ggplot(df, aes({{ xvar }}, prop)) +
      geom_col(color = "black") +
      geom_text(aes(label = n), vjust = 1.3, colour = "white") +
      facet_grid(rows = vars({{ facetvar }}), cols = vars(stage)) +
      labs(title = title)+
      scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.05)))
    if (reef) {
      p <- p +
        scale_x_discrete(limits = REEF_LEVELS4, drop = FALSE) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1))
    }
    p
  }

  # data
  exp4_loc    <- mk_props(df_exp4_brick_long, location_code, tank)
  exp4_round  <- mk_props(df_exp4_brick_long, location_code, round)
  exp4_loc4   <- mk_props(df_exp4_brick_long, reef_type,     tank)
  exp4_round4 <- mk_props(df_exp4_brick_long, reef_type,     round)

  # plots
  p_exp4_loc    <- mk_bar(exp4_loc,   location_code, tank,  "Location by tank")
  p_exp4_round  <- mk_bar(exp4_round, location_code, round, "Location by round")
  p_exp4_loc4   <- mk_bar(exp4_loc4,  reef_type,     tank,  "Reeftype by tank",  reef = TRUE)
  p_exp4_round4 <- mk_bar(exp4_round4,reef_type,     round, "Reeftype by round", reef = TRUE)

  loc_round_plots_exp4 <<- p_exp4_loc + p_exp4_loc4 + p_exp4_round + p_exp4_round4 +
    patchwork::plot_layout(ncol = 4)
}))

loc_round_plots_exp4

```

### Exp 4 Statistic
quarto-executable-code-5450563D

```r
#| label: Statistics Exp 4
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

# RQ1: Is there a statistical difference between reef type in L1 between the size classes while controlling for group_id, tank and round?
# H0:
# H1: 

# RQ2: 
# H0:
# H1: 

df4 <- df_exp4_brick_long %>%
  mutate(
    reef_type      = factor(reef_type),
    stage          = factor(stage, levels = c("L0", "L1")),
    sex            = factor(sex),
    size_class     = factor(size_class, levels = c("S", "M", "L")),
    location_code  = factor(location_code),
    tank           = factor(tank),
    round          = factor(round),
    group_id       = factor(group_id),
    koura_id       = factor(koura_id)) %>%
  filter(!is.na(reef_type))

## Set BrickArtificial as reference so coefficients are for BrickWood vs BrickArtificial
df4 <- df4 %>% mutate(reef_type = relevel(reef_type, ref = "BrickArtificial"))

m_exp4 <- glmer(
  reef_type ~ location_code + sex + size_class + (1 | group_id) + (1 | koura_id) + (1 | tank) + (1 | round),
  data    = df4,
  family  = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa"))

summary(m_exp4)

```

## Waterquality
quarto-executable-code-5450563D

```r
#| label: WQ
#| fig-width: 6
#| fig-height: 7
#| dpi: 120
#| fig-cap: "."

wq_clean <- df_wq %>%
  mutate(value = suppressWarnings(as.numeric(value)),tank  = factor(tank),param_label = ifelse(is.na(unit) | unit %in% c("", "NA"), parameter_abb, paste(parameter_abb, unit))) %>%
  filter(!is.na(tank_use), !is.na(value),data_time >= as.POSIXct("2025-09-01"),tank_use == "Experiment")

wq_probe <- wq_clean %>%
  transmute(datetime = data_time,tank,param_label,value)

# --- Logger data (df_loggers) -> give unique param_label names ---
wq_logger <- df_loggers %>%
  rename(datetime = data_time) %>%
  mutate(tank     = factor(tank),
    datetime = lubridate::ymd_hms(datetime, tz = "Pacific/Auckland")) %>%
  pivot_longer(c(temperature, light), names_to = "param", values_to = "value") %>%
  mutate(
    param_label = case_when(
      param == "temperature" ~ "Temp (°C) - Logger",
      param == "light"       ~ "Light (lux) - Logger",
      TRUE                   ~ param),
    date  = as.Date(datetime),
    time  = format(datetime, "%H:%M:%S"),
    year  = year(datetime),
    month = month(datetime, label = TRUE),
    day   = day(datetime)) %>%
  filter(
    datetime >= as.POSIXct("2025-09-01", tz = "Pacific/Auckland"),
    (param_label != "Light (lux) - Logger" | value <= 1000),
    (param_label != "Temp (°C) - Logger"   | value <= 20) ) %>%
  transmute(datetime, date, time, year, month, day, tank, param_label, value)

# --- Combine into ONE df, keep labels distinct ---
PARAM_LEVELS <- c("Temp (°C)", "Temp (°C) - Logger","Light (lux) - Logger",
                  "Press (mbars)","DO (%)","DO (mg/L)","SPC (µS/cm)","Cond (µS/cm)","SAL (ppt)","pH")

wq_all <- bind_rows(wq_probe, wq_logger) %>%
  mutate(param_label = factor(param_label, levels = PARAM_LEVELS),tank = factor(tank)) %>%
  tidyr::drop_na(value)

# --- Plot from the single df ---
wq_all_plot <- ggplot(wq_all, aes(datetime, value, color = tank, group = tank)) +
  geom_line() +
  facet_wrap(~ param_label, scales = "free_y", ncol = 2)

wq_all_plot
ggsave(wq_all_plot, file = file.path(out_dir, "wq_all_probe_logger.png"), width = 10, height = 8, dpi = 300)

```

# Discussion
## Conclusion


# Acknowledgements

# References

# Supplementary information
## Length-Weight of koura
quarto-executable-code-5450563D

```r
#| label: Length-Weight of koura
#| fig-width: 9
#| fig-height: 5
#| dpi: 300
#| fig-cap: "."

summarise_size_weight <- function(df) {
  df %>%
    summarise(
      n_size          = sum(!is.na(size_mm1)),
      n_weight        = sum(!is.na(weight_g1)),
      size_mm1_mean   = mean(size_mm1, na.rm = TRUE),
      size_mm1_sd     = sd(size_mm1, na.rm = TRUE),
      size_mm1_range  = paste0(min(size_mm1, na.rm = TRUE), "–", max(size_mm1, na.rm = TRUE)),
      weight_g1_mean  = mean(weight_g1, na.rm = TRUE),
      weight_g1_sd    = sd(weight_g1, na.rm = TRUE),
      weight_g1_range = paste0(min(weight_g1, na.rm = TRUE), "–", max(weight_g1, na.rm = TRUE))
    )
}

summarise_size_weight(df_exp1_single_raw)
summarise_size_weight(df_exp2_group_raw)
summarise_size_weight(df_exp3_woodstone_raw)
summarise_size_weight(df_exp4_brick_raw)



df_size_weight_all <- dplyr::bind_rows(
  df_exp1_single_raw %>%
    dplyr::transmute(experiment = "Single",
      size_mm1   = as.numeric(size_mm1),
      weight_g1  = as.numeric(weight_g1),
      sex= as.factor(sex)), df_exp2_group_raw %>%
    dplyr::transmute(
      experiment = "Group",
      size_mm1   = as.numeric(size_mm1),
      weight_g1  = as.numeric(weight_g1),
      sex= as.factor(sex)),df_exp3_woodstone_raw %>%
    dplyr::transmute(
      experiment = "StoneWood",
      size_mm1   = as.numeric(size_mm1),
      weight_g1  = as.numeric(weight_g1),
      sex= as.factor(sex)),df_exp4_brick_raw %>%
    dplyr::transmute(
      experiment = "Brick",
      size_mm1   = as.numeric(size_mm1),
      weight_g1  = as.numeric(weight_g1),
      sex= as.factor(sex))) %>%
  dplyr::filter(!is.na(size_mm1), !is.na(weight_g1)) %>%
  dplyr::mutate(experiment = factor(experiment))

p_size_weight <- ggplot2::ggplot(
  df_size_weight_all,
  ggplot2::aes(x = size_mm1, y = weight_g1, colour = sex)) +
  ggplot2::geom_vline(xintercept = 22, linetype = "dotted") +
  ggplot2::geom_vline(xintercept = 30, linetype = "dotted") +
  ggplot2::geom_point()+
  ggplot2::facet_grid(~ experiment)+
  labs(x = "OCL Length mm", y = "Weight g")

p_size_weight

```
````